# Sistema de Gestión de Citas Médicas — Clínica San Gabriel

### Lenguaje de Programación II — Semestre 2026-2
**Evaluación Grupal 1 · Trabajo Parcial · Grupo 6**

---

| N° | Apellidos y nombres | Usuario de GitHub | Parte |
|---|---|---|---|
| 1 | Anaya Yarleque, Nahomi Alessandra | `@HanakoCrazy` | 1 — `Historial`, `Persona`, `Paciente` |
| 2 | Guerra Vilchez, Ximena Valery | `@ValeryGuerra7` | 2 — `Medico` y polimorfismo |
| 3 | Mendoza Chavez, Sergio | `@290803S` | 3 — `Cita` y reglas de negocio |
| 4 | Fernandez Leyva, Dayvi | `@dferl26` | 4 — `Clinica`, pandas y menú |

---

**Avance del archivo:** ■■■□ &nbsp; partes 1 a 3 de 4 · última parte agregada por **Sergio Mendoza**.

**Siguiente:** Dayvi Fernandez clona el repositorio, agrega la parte 4 al final de este mismo archivo y lo vuelve a subir.

---

### El escenario

La Clínica San Gabriel llevaba el control de sus citas en cuadernos, lo que producía tres problemas: se programaban **dos pacientes con el mismo médico a la misma hora**, el **historial clínico se perdía** y **no existían reportes**.

El sistema resuelve los tres modelando cada elemento del proceso como un objeto: `Paciente`, `Medico`, `Cita`, `Historial` y `Clinica`, todas construidas sobre la clase base `Persona`.

> Ejecute el cuaderno completo con **Entorno de ejecución → Ejecutar todas**.

---

# Parte 1 — Anaya Yarleque, Nahomi Alessandra

Las clases base del sistema. Aquí se definen los datos que comparten todas las personas y la estructura que guarda los antecedentes médicos.

## 1.1 Clase `Historial`

Guarda las atenciones de un paciente. Cada atención es un **diccionario**, para poder convertirlas después en una tabla de pandas sin trabajo adicional.

Se define primero porque `Paciente` la necesita: al crear un paciente se le crea también su historial.

In [1]:
# ------------------------------------------------------------
#  Clase Historial
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Guarda las atenciones de UN paciente. Cada atencion se guarda
# como un diccionario, para poder convertirlas despues en una
# tabla de pandas sin trabajo adicional.


class Historial:
    # Atributo de CLASE: el mismo para todos los historiales
    centro = "Clinica San Gabriel"

    def __init__(self, codigo_paciente):
        # Atributos de INSTANCIA: distintos en cada objeto
        self.codigo_paciente = codigo_paciente
        self.atenciones = []

    def agregar_atencion(self, fecha, especialidad, medico, diagnostico, tratamiento):
        atencion = {
            "fecha": fecha,
            "especialidad": especialidad,
            "medico": medico,
            "diagnostico": diagnostico,
            "tratamiento": tratamiento
        }
        self.atenciones.append(atencion)

    def total_atenciones(self):
        return len(self.atenciones)

    def ultimo_diagnostico(self):
        if len(self.atenciones) == 0:
            return "Sin atenciones registradas"
        ultima = self.atenciones[-1]
        return ultima["diagnostico"]

    def mostrar(self):
        print("HISTORIAL CLINICO DEL PACIENTE " + self.codigo_paciente)
        if len(self.atenciones) == 0:
            print("   (Sin atenciones registradas)")
        else:
            numero = 1
            for a in self.atenciones:
                print("   " + str(numero) + ". " + a["fecha"] + " | " + a["especialidad"] +
                      " | Dx: " + a["diagnostico"] + " | Tto: " + a["tratamiento"])
                numero = numero + 1
        print("   Total de atenciones: " + str(self.total_atenciones()))


print("Clase Historial creada.")

Clase Historial creada.


## 1.2 Clase base `Persona`

Reúne los cuatro datos que comparten pacientes y médicos. Se usan **getters** para leer y **setters** para modificar: el setter revisa el dato antes de guardarlo, por eso `set_telefono("123")` no modifica nada y avisa del error.

El método `mostrar_informacion()` se escribe aquí de forma general y las clases hijas lo van a **sobrescribir**.

In [2]:
# ------------------------------------------------------------
#  Clase base Persona
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Reune los datos que comparten pacientes y medicos. De ella van
# a HEREDAR las clases Paciente y Medico.


class Persona:
    def __init__(self, nombre, apellido, dni, telefono):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni
        self.telefono = telefono

    # ---------- Getters: leer los datos ----------
    def get_nombre_completo(self):
        return self.nombre + " " + self.apellido

    def get_dni(self):
        return self.dni

    # ---------- Setter: modificar con validacion ----------
    def set_telefono(self, nuevo_telefono):
        """Cambia el telefono solo si tiene 9 digitos numericos."""
        if len(nuevo_telefono) == 9 and nuevo_telefono.isdigit():
            self.telefono = nuevo_telefono
            print("Telefono actualizado a " + nuevo_telefono)
        else:
            print("ERROR: el telefono debe tener 9 digitos numericos.")

    # ---------- Metodo que las clases hijas van a sobrescribir ----------
    def mostrar_informacion(self):
        return "[PERSONA]  " + self.get_nombre_completo() + " | DNI: " + self.dni

    def __str__(self):
        return self.get_nombre_completo()


print("Clase Persona creada.")

Clase Persona creada.


## 1.3 Clase `Paciente` (hereda de `Persona`)

Tres mecanismos trabajando en el mismo constructor:

1. **`super().__init__(...)`** llama al constructor del padre. Sin esa línea habría que repetir las cuatro asignaciones de los datos personales en cada clase hija.
2. **`contador`** es un atributo de clase que se incrementa cada vez que nace un paciente, y genera los códigos `P001`, `P002`, `P003`.
3. **`self.historial = Historial(...)`** es **composición**: un objeto que vive dentro de otro.

In [3]:
# ------------------------------------------------------------
#  Clase Paciente (hereda de Persona)
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------


class Paciente(Persona):
    # Atributo de CLASE: genera los codigos P001, P002, P003...
    contador = 0

    def __init__(self, nombre, apellido, dni, telefono, edad, tiene_seguro=False):
        # super() reutiliza el constructor del padre
        super().__init__(nombre, apellido, dni, telefono)

        self.edad = edad
        self.tiene_seguro = tiene_seguro

        Paciente.contador = Paciente.contador + 1
        self.codigo = "P" + str(Paciente.contador).zfill(3)

        # COMPOSICION: cada paciente nace con su propio historial
        self.historial = Historial(self.codigo)

    def get_codigo(self):
        return self.codigo

    def set_edad(self, nueva_edad):
        if nueva_edad > 0 and nueva_edad < 120:
            self.edad = nueva_edad
        else:
            print("ERROR: la edad debe estar entre 1 y 119 anios.")

    def es_adulto_mayor(self):
        """Los pacientes de 60 anios a mas tienen atencion preferente."""
        return self.edad >= 60

    def a_diccionario(self):
        """Convierte el objeto en un diccionario, para usarlo con pandas."""
        return {
            "codigo": self.codigo,
            "nombre": self.get_nombre_completo(),
            "edad": self.edad,
            "tiene_seguro": self.tiene_seguro,
            "atenciones": self.historial.total_atenciones()
        }

    # ---------- Sobrescritura (override) ----------
    def mostrar_informacion(self):
        if self.tiene_seguro:
            seguro = "SI"
        else:
            seguro = "NO"

        texto = ("[PACIENTE] " + self.codigo + " | " + self.get_nombre_completo() +
                 " | Edad: " + str(self.edad) + " | Seguro: " + seguro)

        if self.es_adulto_mayor():
            texto = texto + " | Atencion preferente"
        return texto

    def __str__(self):
        return self.codigo + " - " + self.get_nombre_completo()


print("Clase Paciente creada.")

Clase Paciente creada.


## 1.4 Prueba de la parte 1

In [4]:
# ------------------------------------------------------------
#  Prueba de la parte 1
# ------------------------------------------------------------

print("=" * 58)
print("PRUEBA DE LA PARTE 1")
print("=" * 58)

# ---------- Creacion de objetos ----------
print("\n1. PACIENTES REGISTRADOS")
print("-" * 58)
p1 = Paciente("Nahomi", "Anaya Yarleque", "72345678", "900111222", 23, True)
p2 = Paciente("Ximena", "Guerra Vilchez", "73456789", "900333444", 24)
p3 = Paciente("Luis", "Ramos Diaz", "74567890", "900555666", 67, True)

for paciente in [p1, p2, p3]:
    print(paciente.mostrar_informacion())

# ---------- Atributo de clase ----------
print("\n2. ATRIBUTO DE CLASE")
print("-" * 58)
print("Pacientes creados (Paciente.contador):", Paciente.contador)
print("Codigo del primero (atributo de instancia):", p1.get_codigo())

# ---------- Setters con validacion ----------
print("\n3. SETTERS CON VALIDACION")
print("-" * 58)
p1.set_telefono("123")
p1.set_telefono("955123456")
p2.set_edad(200)

# ---------- Herencia ----------
print("\n4. HERENCIA")
print("-" * 58)
print("isinstance(p1, Paciente):", isinstance(p1, Paciente))
print("isinstance(p1, Persona) :", isinstance(p1, Persona))
print("get_dni() esta escrito en Persona y el paciente lo usa:", p1.get_dni())

# ---------- Composicion ----------
print("\n5. COMPOSICION: CADA PACIENTE TIENE SU HISTORIAL")
print("-" * 58)
p1.historial.mostrar()
print("\nSe agrega una atencion de prueba:")
p1.historial.agregar_atencion("20/10/2026", "Medicina General",
                              "Carlos Rojas", "Faringitis aguda", "Amoxicilina 500mg")
p1.historial.mostrar()

PRUEBA DE LA PARTE 1

1. PACIENTES REGISTRADOS
----------------------------------------------------------
[PACIENTE] P001 | Nahomi Anaya Yarleque | Edad: 23 | Seguro: SI
[PACIENTE] P002 | Ximena Guerra Vilchez | Edad: 24 | Seguro: NO
[PACIENTE] P003 | Luis Ramos Diaz | Edad: 67 | Seguro: SI | Atencion preferente

2. ATRIBUTO DE CLASE
----------------------------------------------------------
Pacientes creados (Paciente.contador): 3
Codigo del primero (atributo de instancia): P001

3. SETTERS CON VALIDACION
----------------------------------------------------------
ERROR: el telefono debe tener 9 digitos numericos.
Telefono actualizado a 955123456
ERROR: la edad debe estar entre 1 y 119 anios.

4. HERENCIA
----------------------------------------------------------
isinstance(p1, Paciente): True
isinstance(p1, Persona) : True
get_dni() esta escrito en Persona y el paciente lo usa: 72345678

5. COMPOSICION: CADA PACIENTE TIENE SU HISTORIAL
-------------------------------------------------

---

# Parte 2 — Guerra Vilchez, Ximena Valery

## 2.1 Clase `Medico` (hereda de `Persona`)

Agrega especialidad, tarifa y **agenda**. La agenda es una lista de textos como `"26/10/2026 09:00"`, y preguntar si el médico está libre es preguntar si ese texto está en la lista:

```python
def esta_disponible(self, fecha, hora):
    horario = fecha + " " + hora
    return horario not in self.agenda
```

Son dos líneas, pero son las que cumplen el **valor añadido** del enunciado: evitar que se agenden dos citas al mismo médico en la misma hora.

In [5]:
# ------------------------------------------------------------
#  Clase Medico (hereda de Persona)
#  Integrante: Guerra Vilchez, Ximena Valery
# ------------------------------------------------------------
# Ademas de los datos personales guarda su AGENDA, que es la
# lista de horarios ya ocupados. Es la base del valor anadido.


class Medico(Persona):
    # Atributo de CLASE: genera los codigos M001, M002, M003...
    contador = 0

    def __init__(self, nombre, apellido, dni, telefono, especialidad, tarifa):
        super().__init__(nombre, apellido, dni, telefono)

        self.especialidad = especialidad
        self.tarifa = tarifa
        self.agenda = []          # lista de horarios reservados

        Medico.contador = Medico.contador + 1
        self.codigo = "M" + str(Medico.contador).zfill(3)

    # ---------- Getters ----------
    def get_codigo(self):
        return self.codigo

    def get_especialidad(self):
        return self.especialidad

    def get_tarifa(self):
        return self.tarifa

    # ---------- Setter ----------
    def set_tarifa(self, nueva_tarifa):
        if nueva_tarifa > 0:
            self.tarifa = nueva_tarifa
            print("Tarifa actualizada a S/ " + str(nueva_tarifa))
        else:
            print("ERROR: la tarifa debe ser mayor que cero.")

    # ---------- La agenda: el valor anadido del proyecto ----------
    def esta_disponible(self, fecha, hora):
        """Responde si el medico tiene libre ese dia y esa hora."""
        horario = fecha + " " + hora
        return horario not in self.agenda

    def ocupar_horario(self, fecha, hora):
        horario = fecha + " " + hora
        if self.esta_disponible(fecha, hora):
            self.agenda.append(horario)
            return True
        else:
            print("ERROR: el Dr(a). " + self.get_nombre_completo() +
                  " ya tiene una cita el " + horario + ".")
            return False

    def liberar_horario(self, fecha, hora):
        horario = fecha + " " + hora
        if horario in self.agenda:
            self.agenda.remove(horario)
            return True
        return False

    def total_citas(self):
        return len(self.agenda)

    # ---------- Sobrescritura (override) ----------
    def mostrar_informacion(self):
        return ("[MEDICO]   " + self.codigo + " | Dr(a). " + self.get_nombre_completo() +
                " | " + self.especialidad + " | S/ " + str(self.tarifa) +
                " | Citas: " + str(self.total_citas()))

    def __str__(self):
        return self.codigo + " - Dr(a). " + self.get_nombre_completo() + " (" + self.especialidad + ")"


print("Clase Medico creada.")

Clase Medico creada.


## 2.2 Prueba de la parte 2

Incluye la demostración de **polimorfismo**: una sola lista guarda pacientes y médicos, y al llamar al mismo método cada objeto responde según su clase, sin un solo `if` de tipo.

In [6]:
# ------------------------------------------------------------
#  Prueba de la parte 2
# ------------------------------------------------------------

print("=" * 58)
print("PRUEBA DE LA PARTE 2")
print("=" * 58)

# ---------- Creacion de medicos ----------
print("\n1. MEDICOS REGISTRADOS")
print("-" * 58)
m1 = Medico("Carlos", "Rojas Vega", "40112233", "987654321", "Medicina General", 60)
m2 = Medico("Lucia", "Paredes Soto", "41223344", "987111222", "Pediatria", 90)
m3 = Medico("Javier", "Quispe Luna", "42334455", "987333444", "Cardiologia", 150)

for medico in [m1, m2, m3]:
    print(medico.mostrar_informacion())

# ---------- Setter con validacion ----------
print("\n2. SETTER CON VALIDACION")
print("-" * 58)
m1.set_tarifa(-50)
m1.set_tarifa(70)

# ---------- El valor anadido ----------
print("\n3. CONTROL DE LA AGENDA (valor anadido)")
print("-" * 58)
print("Libre el 26/10/2026 a las 09:00?:", m1.esta_disponible("26/10/2026", "09:00"))

print("\nSe reserva ese horario:")
m1.ocupar_horario("26/10/2026", "09:00")
print("Libre ahora?:", m1.esta_disponible("26/10/2026", "09:00"))

print("\nSe intenta reservar el MISMO horario otra vez:")
m1.ocupar_horario("26/10/2026", "09:00")

print("\nSe libera el horario:")
m1.liberar_horario("26/10/2026", "09:00")
print("Libre de nuevo?:", m1.esta_disponible("26/10/2026", "09:00"))

# ---------- Polimorfismo ----------
print("\n4. POLIMORFISMO")
print("-" * 58)
print("Una sola lista con pacientes y medicos. Todos responden al")
print("mismo metodo mostrar_informacion(), cada uno a su manera:\n")

personas = [p1, m1, p3, m3]
for elemento in personas:
    print(type(elemento).__name__.ljust(9), "->", elemento.mostrar_informacion())

print("\nNo hizo falta ningun 'if' para saber de que tipo era cada objeto.")

PRUEBA DE LA PARTE 2

1. MEDICOS REGISTRADOS
----------------------------------------------------------
[MEDICO]   M001 | Dr(a). Carlos Rojas Vega | Medicina General | S/ 60 | Citas: 0
[MEDICO]   M002 | Dr(a). Lucia Paredes Soto | Pediatria | S/ 90 | Citas: 0
[MEDICO]   M003 | Dr(a). Javier Quispe Luna | Cardiologia | S/ 150 | Citas: 0

2. SETTER CON VALIDACION
----------------------------------------------------------
ERROR: la tarifa debe ser mayor que cero.
Tarifa actualizada a S/ 70

3. CONTROL DE LA AGENDA (valor anadido)
----------------------------------------------------------
Libre el 26/10/2026 a las 09:00?: True

Se reserva ese horario:
Libre ahora?: False

Se intenta reservar el MISMO horario otra vez:
ERROR: el Dr(a). Carlos Rojas Vega ya tiene una cita el 26/10/2026 09:00.

Se libera el horario:
Libre de nuevo?: True

4. POLIMORFISMO
----------------------------------------------------------
Una sola lista con pacientes y medicos. Todos responden al
mismo metodo mostrar_i

---

# Parte 3 — Mendoza Chavez, Sergio

## 3.1 Clase `Cita`

Relaciona un `Paciente` con un `Medico`. Guarda los **objetos completos**, no sus códigos, para poder preguntarles directamente: `self.medico.get_tarifa()`.

Sus tres atributos de clase (`contador`, `descuento_seguro`, `estados`) valen lo mismo para todas las citas: si la clínica cambia el descuento al 25 %, se modifica una línea.

### Ciclo de vida de una cita

```
                  atender()
  PENDIENTE  ──────────────────▶  ATENDIDA    (escribe en el historial)
      │ cancelar()
      ▼
  CANCELADA    (libera el horario del médico)
```

Las dos transiciones son definitivas: los métodos revisan el estado actual antes de cambiar nada, por eso una cita no se puede atender dos veces ni cancelar después de atendida.

El método `atender()` hace tres cosas en un solo llamado: cambia el estado, guarda el diagnóstico y **escribe en el historial del paciente**. Gracias a lo último es imposible atender una cita y olvidar el registro.

In [2]:
# ------------------------------------------------------------
#  Clase Cita
#  Integrante: Mendoza Chavez, Sergio
# ------------------------------------------------------------
# Relaciona un objeto Paciente con un objeto Medico en una fecha
# y una hora. Es la clase que une todo el sistema.


class Cita:
    # ---------- ATRIBUTOS DE CLASE ----------
    contador = 0
    descuento_seguro = 0.20
    estados = ["PENDIENTE", "ATENDIDA", "CANCELADA"]

    def __init__(self, paciente, medico, fecha, hora, motivo):
        Cita.contador = Cita.contador + 1
        self.codigo = "C" + str(Cita.contador).zfill(3)

        # Se guardan los OBJETOS completos, no solo sus codigos
        self.paciente = paciente
        self.medico = medico

        self.fecha = fecha
        self.hora = hora
        self.motivo = motivo

        self.estado = "PENDIENTE"      # toda cita nueva nace pendiente
        self.diagnostico = ""
        self.tratamiento = ""

    def get_codigo(self):
        return self.codigo

    def get_estado(self):
        return self.estado

    def costo(self):
        """Le pregunta la tarifa al medico y el seguro al paciente."""
        precio = self.medico.get_tarifa()
        if self.paciente.tiene_seguro:
            precio = precio - (precio * Cita.descuento_seguro)
        return round(precio, 2)

    def atender(self, diagnostico, tratamiento):
        """Cambia el estado y escribe en el historial del paciente."""
        if self.estado != "PENDIENTE":
            print("ERROR: la cita " + self.codigo + " ya fue " + self.estado.lower() + ".")
            return False

        if len(diagnostico) < 4:
            print("ERROR: el diagnostico es demasiado corto.")
            return False

        self.diagnostico = diagnostico
        self.tratamiento = tratamiento
        self.estado = "ATENDIDA"

        # Escribe en el historial del paciente
        self.paciente.historial.agregar_atencion(
            self.fecha,
            self.medico.get_especialidad(),
            self.medico.get_nombre_completo(),
            diagnostico,
            tratamiento
        )

        print("La cita " + self.codigo + " fue atendida y se registro en el historial.")
        return True

    def cancelar(self):
        """Cancela la cita y libera el horario del medico."""
        if self.estado == "ATENDIDA":
            print("ERROR: no se puede cancelar una cita que ya fue atendida.")
            return False
        if self.estado == "CANCELADA":
            print("ERROR: la cita " + self.codigo + " ya estaba cancelada.")
            return False

        self.estado = "CANCELADA"
        self.medico.liberar_horario(self.fecha, self.hora)
        print("La cita " + self.codigo + " fue cancelada y el horario quedo libre.")
        return True

    def comprobante(self):
        if self.estado != "ATENDIDA":
            return "La cita " + self.codigo + " no genera comprobante (estado: " + self.estado + ")."

        linea = "+" + "-" * 48 + "+"
        t = linea + "\n"
        t = t + "|          COMPROBANTE DE ATENCION               |\n"
        t = t + linea + "\n"
        t = t + " Cita       : " + self.codigo + "\n"
        t = t + " Fecha      : " + self.fecha + "   " + self.hora + "\n"
        t = t + " Paciente   : " + self.paciente.get_nombre_completo() + "\n"
        t = t + " Medico     : Dr(a). " + self.medico.get_nombre_completo() + "\n"
        t = t + " Servicio   : " + self.medico.get_especialidad() + "\n"
        t = t + " Diagnostico: " + self.diagnostico + "\n"
        t = t + " " + "-" * 47 + "\n"
        t = t + " Tarifa     : S/ " + str(self.medico.get_tarifa()) + "\n"
        if self.paciente.tiene_seguro:
            descuento = self.medico.get_tarifa() * Cita.descuento_seguro
            t = t + " Seguro -20%: S/ " + str(round(descuento, 2)) + "\n"
        t = t + " TOTAL      : S/ " + str(self.costo()) + "\n"
        t = t + linea
        return t

    def a_diccionario(self):
        """Convierte la cita en un diccionario, para usarlo con pandas."""
        return {
            "codigo": self.codigo,
            "fecha": self.fecha,
            "paciente": self.paciente.get_nombre_completo(),
            "medico": self.medico.get_nombre_completo(),
            "especialidad": self.medico.get_especialidad(),
            "estado": self.estado,
            "costo": self.costo()
        }

    def __str__(self):
        return (self.codigo + " | " + self.fecha + " " + self.hora +
                " | " + self.paciente.get_nombre_completo() +
                " | Dr(a). " + self.medico.get_nombre_completo() +
                " | " + self.estado)


print("Clase Cita creada.")

Clase Cita creada.
